In [1]:
import numpy as np
import pandas as pd

In [2]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data = pd.read_stata(r"datos/ECU_1997m11_BID.dta") # para bases de stata

## Revisar los datos

- rn - región natural
- dominio - dominio
- cuidad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- persona - persona
- numpers - número de personas
- edad - edad
- ingpat - Ingresos como patrono o cuenta propia
- ingasg - Ingresos como asalariado de gobierno
- ingepv - Ingresos asalariado empresa privada
- ingdom - Ingresos como empleada doméstica
- ingalq - Ingresos por alquileres, rentas o interese
- ingjub - Ingresos por jubilación o pensión
- ingotr - por otros ingresos
- fexp - factor de expansión
- ingrl - ingresos

El valor de 'ingasg' e 'ingepv' son el ingreso laboral monetario vamos a incluir 'ingdom' para darle un scope mayor, no hay datos sobre ingreso laboral no monetario, las otras variables son ingreso no laboral monetario y no monetario e ingrl es un ingreso total

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36684 entries, 0 to 36683
Columns: 251 entries, region_BID_c to ppp_wdi2011
dtypes: category(77), float32(18), float64(105), int16(7), int32(1), int8(33), object(10)
memory usage: 39.2+ MB


Filtramos solo las columnas de interés para alivar el peso en la memoria

In [4]:
data.columns

Index(['region_BID_c', 'region_c', 'pais_c', 'anio_c', 'mes_c', 'zona_c',
       'factor_ch', 'idh_ch', 'idp_ci', 'factor_ci',
       ...
       'formal', 'formal_1', 'ylm_ci1', 'ynlm_ci1', 'aguafconsumo_ch',
       'aguafuente_ch', 'aguadisp1_ch', 'aguadisp2_ch', 'sinbano_ch',
       'ppp_wdi2011'],
      dtype='object', length=251)

- rn - regiÓn natural
- dominio - dominio
- ciudad - ciudad
- zona - zona
- sector - sector
- vivienda - vivienda
- hogar - hogar
- formul - formulario
- persona - persona
- numpers - nÚmero de personas
- edad - edad
- sexo - sexo
- iess - es afiliado al iess
- trabajo - trabajó la semana pasada
- rama - rama de actividad
- grupo - grupo de ocupación
- catetrab - categoría de ocupación
- ramas - rama de actividad secundaria
- grupos - grupo de ocupación secundario
- cates - categoría de ocupación secundaria
- ingpat - ingresos como patrono o cuenta propia
- ingasg - ingresos como asalariado de gobierno
- ingepv - ingreso asalariado empresa privada
- ingdom - ingreso como empleada doméstica
- ingjub - ingresos por jubilación o pensión
- ingalq - ingreso por alquileres, rentas o intereses
- ingotr - por otros ingresos
- ingrl - ingresos
- fexp1 - factor de expanción 1
- fexp - factor de expanción

In [5]:
data = data[['rn', 'dominio', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar',
       'formul', 'persona', 'numpers', 'edad', 'trabajo', 'ingpat', 'ingasg',
       'ingepv', 'ingdom', 'ingjub', 'ingalq', 'ingotr', 'nov', 'dic', 'ene',
       'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'ingrl', 'fexp1', 'fexp']]

In [5]:
data ['sum_ing'] = data [['ingasg', 'ingepv', 'ingdom']].sum(axis=1)

In [6]:
data['suma_ing_ene'] = data.apply(lambda x: x['sum_ing'] if x['ene'] == 'trabajando' else None, axis=1)
data['suma_ing_feb'] = data.apply(lambda x: x['sum_ing'] if x['feb'] == 'trabajando' else None, axis=1)
data['suma_ing_mar'] = data.apply(lambda x: x['sum_ing'] if x['mar'] == 'trabajando' else None, axis=1)
data['suma_ing_abr'] = data.apply(lambda x: x['sum_ing'] if x['abr'] == 'trabajando' else None, axis=1)
data['suma_ing_may'] = data.apply(lambda x: x['sum_ing'] if x['may'] == 'trabajando' else None, axis=1)
data['suma_ing_jun'] = data.apply(lambda x: x['sum_ing'] if x['jun'] == 'trabajando' else None, axis=1)
data['suma_ing_jul'] = data.apply(lambda x: x['sum_ing'] if x['jul'] == 'trabajando' else None, axis=1)
data['suma_ing_ago'] = data.apply(lambda x: x['sum_ing'] if x['ago'] == 'trabajando' else None, axis=1)
data['suma_ing_sep'] = data.apply(lambda x: x['sum_ing'] if x['sep'] == 'trabajando' else None, axis=1)
data['suma_ing_oct'] = data.apply(lambda x: x['sum_ing'] if x['oct'] == 'trabajando' else None, axis=1)
data['suma_ing_nov'] = data.apply(lambda x: x['sum_ing'] if x['nov'] == 'trabajando' else None, axis=1)
data['suma_ing_dic'] = data.apply(lambda x: x['sum_ing'] if x['dic'] == 'trabajando' else None, axis=1)

In [6]:
data['suma_ing_t1'] = (data['suma_ing_ene'] + data['suma_ing_feb'] + data['suma_ing_mar'])/3
data['suma_ing_t2'] = (data['suma_ing_abr'] + data['suma_ing_may'] + data['suma_ing_jun'])/3
data['suma_ing_t3'] = (data['suma_ing_jul'] + data['suma_ing_ago'] + data['suma_ing_sep'])/3
data['suma_ing_t4'] = (data['suma_ing_oct'] + data['suma_ing_nov'] + data['suma_ing_dic'])/3

In [7]:
# Llenar valores perdidos con un dato (0)
data['suma_ing_t1'] = data['suma_ing_t1'].fillna(0)
data['suma_ing_t2'] = data['suma_ing_t2'].fillna(0)
data['suma_ing_t3'] = data['suma_ing_t3'].fillna(0)
data['suma_ing_t4'] = data['suma_ing_t4'].fillna(0)

In [9]:
data 

,rn,dominio,ciudad,zona,sector,vivienda,hogar,formul,persona,numpers,...,suma_ing_jul,suma_ing_ago,suma_ing_sep,suma_ing_oct,suma_ing_nov,suma_ing_dic,suma_ing_t1,suma_ing_t2,suma_ing_t3,suma_ing_t4
0,1,6,100450,004,008,01,1,1,1,3,...,9999999.0,9999999.0,9999999.0,9999999.0,9999999.0,9999999.0,9999999.0,9999999.0,9999999.0,9999999.0
1,1,6,100450,004,008,01,1,1,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,1,6,100450,004,008,01,1,1,3,3,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
3,1,6,100450,004,008,02,1,1,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,1,6,100450,004,008,02,1,1,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36679,2,2,090150,179,008,12,1,1,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
36680,2,2,090150,179,008,12,1,1,4,4,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
36681,2,7,090750,013,004,01,1,1,4,4,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
36682,2,6,091550,004,004,11,1,1,7,10,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0


In [10]:
data['suma_ing_t1'].mean()

np.float64(281251.6249591102)

In [8]:
pip install openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import openpyxl
print(openpyxl.__version__)

3.1.5


In [16]:
Esto deja todo en dólares constantes de 2014, utilizamos el IPC de Estados Unidos para ajustar por inflación ya que no podemos usar la inflación en sucres si queremos dejar el valor final en dólares de 2014

In [9]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 1997]
datos_base = data_externa[data_externa['Año'] == 2014]

In [19]:
datos_actual

,Año,trimestre,Nacional,Sierra,Costa,Guayaquil,Esmeraldas,Machala,Manta,Santo Domingo,Quito,Loja,Cuenca,Ambato,salario básico unificado,tipo de cambio,umbral de pobreza
28,1997,1,9.57592,NaN,NaN,11.043915,NaN,NaN,NaN,NaN,9.896307,NaN,8.005899,NaN,23.757127,3756.000000,493.696921
29,1997,2,9.57592,NaN,NaN,11.043915,NaN,NaN,NaN,NaN,9.896307,NaN,8.005899,NaN,25.007502,3924.000000,497.666602
30,1997,3,9.57592,NaN,NaN,11.043915,NaN,NaN,NaN,NaN,9.896307,NaN,8.005899,NaN,25.007502,4098.333333,501.673280
31,1997,4,9.57592,NaN,NaN,11.043915,NaN,NaN,NaN,NaN,9.896307,NaN,8.005899,NaN,25.007502,4340.333333,505.894834


In [10]:
ipc_dict = dict(zip(datos_actual['trimestre'], datos_actual['IPC Estados Unidos']))

ipc_base_dict = dict(zip(datos_base['trimestre'], datos_base['IPC Estados Unidos']))

tipo_cambio_dict = dict(zip(datos_actual['trimestre'], datos_actual['tipo de cambio']))

In [21]:
### Asignamos el ipc y tipo de cambio correspondiente según trimestre

{1: 3756.0, 2: 3924.0, 3: 4098.33333333333, 4: 4340.33333333333}

In [25]:
$\begin{equation}
    ingr_{USD-base-2014}^{i} = \frac{ingr_{sucres}^{i}}{tipo-de-cambio^{i}}\left( \frac{ipcUSA^{i}_{2014}}{ipcUSA^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [11]:
data['ipc_t1'] = ipc_dict.get(1)
data['ipc_base_t1'] = ipc_base_dict.get(1)
data['tipo_cambio_t1'] = tipo_cambio_dict.get(1)

data['ipc_t2'] = ipc_dict.get(2)
data['ipc_base_t2'] = ipc_base_dict.get(2)
data['tipo_cambio_t2'] = tipo_cambio_dict.get(2)

data['ipc_t3'] = ipc_dict.get(3)
data['ipc_base_t3'] = ipc_base_dict.get(3)
data['tipo_cambio_t3'] = tipo_cambio_dict.get(3)

data['ipc_t4'] = ipc_dict.get(4)
data['ipc_base_t4'] = ipc_base_dict.get(4)
data['tipo_cambio_t4'] = tipo_cambio_dict.get(4)

In [12]:
# Calculamos el deflactor
data['def'] = (data['ipc_base'] / data['ipc'])

In [33]:
data['def']

0        10.242675
1              NaN
2              NaN
3              NaN
4              NaN
           ...    
36679          NaN
36680          NaN
36681          NaN
36682          NaN
36683          NaN
Name: def, Length: 36684, dtype: float64

In [13]:
data['ingr_t1_r'] = (data['suma_ing_t1'] / data['tipo_cambio']) * data['def']
data['ingr_t2_r'] = (data['suma_ing_t2'] / data['tipo_cambio']) * data['def']
data['ingr_t3_r'] = (data['suma_ing_t3'] / data['tipo_cambio']) * data['def']
data['ingr_t4_r'] = (data['suma_ing_t4'] / data['tipo_cambio']) * data['def']

In [35]:
data[['suma_ing_t1', 'suma_ing_t2', 'suma_ing_t3', 'suma_ing_t4', 'ipc', 'ipc_base', 'def', 'tipo_cambio', 'ingr_t1_r', 'ingr_t2_r', 'ingr_t3_r', 'ingr_t4_r']]

,suma_ing_t1,suma_ing_t2,suma_ing_t3,suma_ing_t4,ipc,ipc_base,def,tipo_cambio,ingr_t1_r,ingr_t2_r,ingr_t3_r,ingr_t4_r
0,9999999.0,9999999.0,9999999.0,9999999.0,9.57592,98.083034,10.242675,3756.0,27270.164227,27270.164227,27270.164227,27270.164227
1,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
36679,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36680,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36681,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36682,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
data['ingr_t1_r'] = (data['ingr_ene_r'] + data['ingr_feb_r'] + data['ingr_mar_r'])/3
data['ingr_t2_r'] = (data['ingr_abr_r'] + data['ingr_may_r'] + data['ingr_jun_r'])/3
data['ingr_t3_r'] = (data['ingr_jul_r'] + data['ingr_ago_r'] + data['ingr_sep_r'])/3
data['ingr_t4_r'] = (data['ingr_oct_r'] + data['ingr_nov_r'] + data['ingr_dic_r'])/3

data['ingr_t4_r'].mean()

In [15]:
columnas_idef = ['rn', 'dominio', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data['idef_hogar'] = data[columnas_idef].astype(str).agg(''.join, axis=1)
len(data['idef_hogar'].unique())

8259

In [16]:
data[['rn', 'dominio', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar', 'idef_hogar', 'persona', 'numpers']]

,rn,dominio,ciudad,zona,sector,vivienda,hogar,idef_hogar,persona,numpers
0,1,3,10150,001,007,01,1,1310150001007011,2,6
1,1,3,10150,001,007,01,1,1310150001007011,4,6
2,1,3,10150,001,007,01,1,1310150001007011,5,6
3,1,3,10150,001,007,01,1,1310150001007011,3,6
4,1,3,10150,001,007,01,1,1310150001007011,6,6
...,...,...,...,...,...,...,...,...,...,...
36679,3,8,210450,001,011,11,1,38210450001011111,3,5
36680,3,8,210450,001,011,11,1,38210450001011111,2,5
36681,3,8,210450,001,011,11,1,38210450001011111,4,5
36682,3,8,210450,001,011,11,1,38210450001011111,1,5


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [17]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [18]:
data['ingr_t1_h'] = data.groupby('idef_hogar')['ingr_t1_r'].transform('sum')
data['ingr_t2_h'] = data.groupby('idef_hogar')['ingr_t2_r'].transform('sum')
data['ingr_t3_h'] = data.groupby('idef_hogar')['ingr_t3_r'].transform('sum')
data['ingr_t4_h'] = data.groupby('idef_hogar')['ingr_t4_r'].transform('sum')

In [19]:
data[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h', 'ingr_t4_h']]

,ingr_t1_h,ingr_t2_h,ingr_t3_h,ingr_t4_h
0,27270.164227,27270.164227,27270.164227,27270.164227
1,27270.164227,27270.164227,27270.164227,27270.164227
2,27270.164227,27270.164227,27270.164227,27270.164227
3,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...
36679,0.000000,0.000000,0.000000,0.000000
36680,0.000000,0.000000,0.000000,0.000000
36681,3076.193196,3076.193196,3076.193196,3076.193196
36682,946.520983,946.520983,946.520983,946.520983


In [20]:
print("Ingreso medio de un hogar t4: ", data['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  418.2311205857671
Mediana del ingreso de un hogar t4:  296.5046807764195


In [21]:
len(data)

36684

In [43]:
data['edad'] = data['edad'].apply(lambda x: x if type(x) == int else 0)

In [22]:
data ['edad']

0        34
1        38
2         6
3        67
4        66
         ..
36679    14
36680     8
36681    15
36682    10
36683    67
Name: edad, Length: 36684, dtype: int64

In [23]:
data = data.loc[(data['edad'] >= 0) & (data['edad'] < 100)]
len(data)

36684

In [24]:
k = 0.4
s = 0.9

In [25]:
# Si es necesario calcular el número de niños
data['es_nino'] = data['edad'] < 10

data['ninos'] = data.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data['es_adulto'] = data['edad'] > 10

data['adultos'] = data.groupby('idef_hogar')['es_adulto'].transform('sum')

In [26]:
data['escala'] = (data['adultos'] + k * data['ninos']) ** s

In [27]:
data['ingr_t_t1'] = data['ingr_t1_h'] / data['escala']
data['ingr_t_t2'] = data['ingr_t2_h'] / data['escala']
data['ingr_t_t3'] = data['ingr_t3_h'] / data['escala']
data['ingr_t_t4'] = data['ingr_t4_h'] / data['escala']

In [28]:
data[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3', 'ingr_t_t4']]

,ingr_t_t1,ingr_t_t2,ingr_t_t3,ingr_t_t4
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
...,...,...,...,...
36679,110.115626,106.224232,101.421379,94.624563
36680,110.115626,106.224232,101.421379,94.624563
36681,110.115626,106.224232,101.421379,94.624563
36682,110.115626,106.224232,101.421379,94.624563


In [29]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  112.42522275699244
Mediana del ingreso individual descontando cargas familiares t4:  79.15454880778373


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

In [52]:
print("Ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  987.7890712278229
Mediana del ingreso individual descontando cargas familiares t4:  407.72516117696057


In [30]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

In [54]:
# Asigna el umbral por trimestre si hay umbral por región modificar
data['umbral'] = data['trimestre'].map(umbral_dict)

data['umbral'] = data.groupby('idef_hogar')['umbral'].transform('mean')

In [31]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [32]:
data['persona_fexp'] = 1 * data['fexp']

In [33]:
for t in [1, 2, 3, 4]:
    condicion = data['trimestre'] == t
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    data.loc[condicion, col_pobres] = (
        (data.loc[condicion, col_ingr] - data.loc[condicion, 'umbral']) < 0
    ).astype(int)

In [34]:
data[['pobres_t1', 'pobres_t2', 'pobres_t3', 'pobres_t4']].sum()

pobres_t1    7893
pobres_t2    8276
pobres_t3    8771
pobres_t4    9699
dtype: int64

In [35]:
print("pobreza t1: ", (data['pobres_t1'] * data['fexp']).sum()/data.loc[data['trimestre'] == 1]['persona_fexp'].sum())
print("pobreza t2: ", (data['pobres_t2'] * data['fexp']).sum()/data.loc[data['trimestre'] == 2]['persona_fexp'].sum())
print("pobreza t3: ", (data['pobres_t3'] * data['fexp']).sum()/data.loc[data['trimestre'] == 3]['persona_fexp'].sum())
print("pobreza t4: ", (data['pobres_t4'] * data['fexp']).sum()/data.loc[data['trimestre'] == 4]['persona_fexp'].sum())

pobreza t1:  0.30514565211846034
pobreza t2:  0.31206790372327475
pobreza t3:  0.33637554512939
pobreza t4:  0.3775450975631619


In [36]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data['trimestre'] == t].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [37]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.305146,0.116295,0.063779,NaN,NaN,NaN,NaN
t2,0.312068,0.122326,0.067303,NaN,NaN,NaN,NaN
t3,0.336376,0.132495,0.072649,NaN,NaN,NaN,NaN
t4,0.377545,0.151276,0.083686,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [38]:
for t in [1, 2, 3, 4]:
    # Filtramos para cada trimestre
    df_temp = data.loc[data['trimestre'] == t].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [39]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.305146,0.116295,0.063779,0.102418,0.191308,0.269375,NaN
t2,0.312068,0.122326,0.067303,0.101481,0.189683,0.267285,NaN
t3,0.336376,0.132495,0.072649,0.10118,0.189121,0.26648,NaN
t4,0.377545,0.151276,0.083686,0.101426,0.189671,0.267376,NaN


Guardamos el ingreso promedio

In [40]:
for t in [1, 2, 3, 4]:
    df_temp = data.loc[data['trimestre'] == t].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada


In [41]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,0.305146,0.116295,0.063779,0.102418,0.191308,0.269375,145.855821
t2,0.312068,0.122326,0.067303,0.101481,0.189683,0.267285,140.926241
t3,0.336376,0.132495,0.072649,0.10118,0.189121,0.26648,134.064311
t4,0.377545,0.151276,0.083686,0.101426,0.189671,0.267376,124.387171


### Inserta los cálculos en la base final

In [42]:
indices = pd.read_csv("indices.csv", encoding='latin-1')

In [43]:
ano = 1997
# Asegurar que el índice de datos_final coincide con trimestres 1..4
datos_final = datos_final.copy()
datos_final["trimestre"] = [1, 2, 3, 4]
datos_final["Año"] = ano

# Reemplazar en indices usando mask
for col in ["fgt0","fgt1","fgt2","a25","a50","a75","ingreso_promedio"]:
    indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values

/tmp/ipykernel_96595/898369095.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.30514565211846034 0.31206790372327475 0.33637554512939
 0.3775450975631619]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
/tmp/ipykernel_96595/898369095.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.1162950529221571 0.12232564311047561 0.13249480645393347
 0.15127576171930474]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
/tmp/ipykernel_96595/898369095.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.0637785328988278 0.06730266458619663 0.07264920951034

In [44]:
datos_final.to_csv('datos_final1997.csv')